In [1]:
# ===== 셀 1: 준비물 가져오기 (도구 불러오기) =====

import os                              # 컴퓨터의 환경설정(.env 값 등)을 읽기 위한 도구
import requests                        # 인터넷 주소에 데이터를 요청(다운로드)하는 도구
import pandas as pd                    # 표(엑셀 같은 형태)로 데이터를 다루는 도구. 보통 'pd'라는 별명으로 씀
from dotenv import load_dotenv         # .env 파일을 읽어오는 도구
# (서울 API는 XML이 아니라 JSON으로 받기 때문에 xml 도구는 필요 없습니다)

load_dotenv()                          # .env 파일을 읽어서 안의 값들을 사용할 수 있게 준비
SERVICE_KEY = os.getenv("DATA_SEOUL_KEY")   # .env 안의 DATA_SEOUL_KEY 값을 꺼내서 변수에 저장

print("서비스 키 로드됨:", bool(SERVICE_KEY))   # 키를 잘 불러왔는지 확인 (True면 성공)

서비스 키 로드됨: True


In [2]:
# ===== 셀 2: 서울시 문화행사 정보 데이터를 인터넷에서 모두 가져오기 =====
# (컬럼명 한글 변환은 하지 않고 원본 컬럼명 그대로 가져옵니다)
#
# 서울 열린데이터광장 API 주소 형식 (KOPIS와 다름! 키와 형식을 '주소 경로'에 직접 넣음):
#   http://openapi.seoul.go.kr:8088/{인증키}/{형식}/{서비스명}/{시작번호}/{끝번호}/

BASE = "http://openapi.seoul.go.kr:8088"   # 서버 기본 주소
SERVICE = "culturalEventInfo"              # 서비스명: 서울시 문화행사 정보
UNIT = 1000                                # 한 번에 가져올 최대 건수 (이 API의 최대값)

rows = []                                  # 가져온 행사들을 담아둘 빈 목록(리스트)
start = 1                                  # 가져올 시작 번호 (1번부터)
total = None                               # 전체 건수 (첫 응답에서 알아냄)

while True:                                # 더 이상 데이터가 없을 때까지 반복
    end = start + UNIT - 1                  # 이번에 가져올 끝 번호
    url = f"{BASE}/{SERVICE_KEY}/json/{SERVICE}/{start}/{end}/"   # 요청 주소 만들기
    res = requests.get(url)                 # 서버에 데이터 요청 → 응답을 res에 저장
    data = res.json()[SERVICE]              # JSON 응답에서 서비스 부분만 꺼냄

    if total is None:                       # 첫 번째 응답이라면
        total = data["list_total_count"]    # 전체 건수를 기억해 둠

    batch = data.get("row", [])             # 이번에 받은 실제 데이터 목록
    rows += batch                           # 전체 목록에 이어 붙이기

    if start + UNIT > total:                # 다음 시작번호가 전체 건수를 넘으면 = 끝
        break                               # 반복을 멈춤
    start += UNIT                           # 아니면 다음 묶음으로 이동

df = pd.DataFrame(rows)                     # 모은 데이터를 표(df)로 만듦 (컬럼명은 원본 그대로)

print(f"서비스: {SERVICE}")                  # 어떤 데이터인지 출력
print(f"전체 건수(API 기준): {total}건")       # API가 알려준 전체 건수
print(f"실제 가져온 데이터: {len(df)}건")        # 우리가 실제로 모은 건수

서비스: culturalEventInfo
전체 건수(API 기준): 3915건
실제 가져온 데이터: 3915건


In [3]:
# ===== 셀 3: 가져온 데이터 확인하기 =====

print("[컬럼 목록]")
print(list(df.columns))               # 어떤 컬럼들이 있는지 (원본 컬럼명) 확인

print("\n[표 크기] (행, 열):", df.shape)   # 데이터가 몇 행 몇 열인지 확인

print("\n[데이터 정보]")
df.info()                             # 각 컬럼의 타입과 빈 값 여부 등 요약 정보

[컬럼 목록]
['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT', 'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK', 'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE', 'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME']

[표 크기] (행, 열): (3915, 24)

[데이터 정보]
<class 'pandas.DataFrame'>
RangeIndex: 3915 entries, 0 to 3914
Data columns (total 24 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CODENAME   3915 non-null   str  
 1   GUNAME     3915 non-null   str  
 2   TITLE      3915 non-null   str  
 3   DATE       3915 non-null   str  
 4   PLACE      3915 non-null   str  
 5   ORG_NAME   3915 non-null   str  
 6   USE_TRGT   3915 non-null   str  
 7   USE_FEE    3915 non-null   str  
 8   INQUIRY    3915 non-null   str  
 9   PLAYER     3915 non-null   str  
 10  PROGRAM    3915 non-null   str  
 11  ETC_DESC   3915 non-null   str  
 12  ORG_LINK   3915 non-null   str  
 13  MAIN_IMG   3915 non-nul

In [4]:
# ===== 셀 4: 상위 10줄 미리보기 =====

df.head(10)   # df 표의 맨 위 10줄을 그대로 보여줌 (모든 컬럼)

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,클래식,영등포구,[영등포문화재단] 마티네콘서트 With 금난새 #3.베버,2026-10-15~2026-10-15,영등포아트홀,영등포문화재단,초등학생(2019년생 포함) 이상,"전석 15,000원",02-2629-2250,,...,2026-05-18,기관,2026-10-15 00:00:00.0,2026-10-15 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,11:00
1,클래식,마포구,[마포문화재단] 2026 M 아티스트 선율 피아노 리사이틀 Ⅱ,2026-09-16~2026-09-16,마포아트센터 아트홀맥,마포문화재단,8세이상 관람가능(미취학아동입장불가),"R석 30,000원, S석 20,000원",02-3274-8600 [문의1번],,...,2026-05-13,기관,2026-09-16 00:00:00.0,2026-09-16 00:00:00.0,기타,126.945533810385,37.5499060881738,무료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,콘서트,마포구,[마포문화재단] M 마티네 [2026 MAC 모닝 콘서트 ＃6],2026-08-26~2026-08-26,마포아트센터 아트홀맥,마포문화재단,8세이상 관람가능(미취학아동입장불가),"전석 20,000원 [패키지권 구매 시 20% 할인]",02-3274-8600 [문의1번],,...,2026-05-13,기관,2026-08-26 00:00:00.0,2026-08-26 00:00:00.0,기타,126.945533810385,37.5499060881738,유료,https://culture.seoul.go.kr/culture/culture/cu...,11:00
3,무용,영등포구,[영등포문화재단] 영등포아트홀 상주단체 '서울발레시어터' [해설이 있는 고전발레],2026-08-22~2026-08-22,영등포아트홀,영등포문화재단,초등학생 이상(2019년생 포함),"전석 20,000원",02-2629-2250,,...,2026-05-18,기관,2026-08-22 00:00:00.0,2026-08-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,16:00
4,전시/미술,강남구,서울일러스트레이션페어V.21,2026-07-30~2026-08-02,서울 코엑스 C홀,기타,누구나,"성인 : 15,000원 / 청소년 : 9,000원",070-7682-6720,,...,2026-04-24,시민,2026-07-30 00:00:00.0,2026-08-02 00:00:00.0,기타,127.059159043842,37.5118239121138,유료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 18:00 *입장마감: 전시 관람 종료 1시간 전
5,무용,강북구,[꿈의숲아트센터] 2026 여름방학 시즌공연 [최태지의 발레 보물상자],2026-07-25~2026-07-25,꿈의숲아트센터 퍼포먼스홀,세종문화회관,"성인, 어린이, 청소년","전석 35,000원",02-399-1000,,...,2026-05-28,기관,2026-07-25 00:00:00.0,2026-07-25 00:00:00.0,어린이/청소년 문화행사,126.9760053,37.5726241,유료,https://culture.seoul.go.kr/culture/culture/cu...,토요일 14:00 / 16:30
6,클래식,마포구,2026 마포문화재단 가족·어린이 축제 [해피 마포 와글와글] 이머시브 뮤지컬 [고...,2026-07-24~2026-08-16,마포아트센터 아트홀맥,마포문화재단,8세이상 관람가능(미취학아동입장불가),"R석 70,000원, S석 50,000원",02-3274-8600 [문의1번],,...,2026-05-09,기관,2026-07-24 00:00:00.0,2026-08-16 00:00:00.0,기타,126.945533810385,37.5499060881738,유료,https://culture.seoul.go.kr/culture/culture/cu...,"11:00, 14:00, 16:30 * 월 공연없음"
7,콘서트,영등포구,[영등포문화재단] 어슬렁 어슬렁 콘서트 #여름 [죠지&김뜻돌],2026-07-24~2026-07-24,영등포아트홀,영등포문화재단,중학생(2013년생 포함) 이상,"전석 45,000원",02-2629-2250,,...,2026-05-18,기관,2026-07-24 00:00:00.0,2026-07-24 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
8,교육/체험,은평구,[은평문화재단] 가가호호 - [춤추는 여름방학],2026-07-18~2026-08-08,구립은평뉴타운도서관 2층 다목적실,은평문화재단,초등학생 자녀와 보호자,,070-4239-9485,,...,2026-06-16,기관,2026-07-18 00:00:00.0,2026-08-08 00:00:00.0,가족 문화행사,126.93311834754995,37.63723039689538,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 12:00
9,연극,강북구,[꿈의숲아트센터] 올리브와 찐콩 [피노키오 트라이얼],2026-07-18~2026-07-19,꿈의숲아트센터 퍼포먼스홀,세종문화회관,초등학생 이상,"전석 20,000원",02-564-5111,,...,2026-06-10,기관,2026-07-18 00:00:00.0,2026-07-19 00:00:00.0,어린이/청소년 문화행사,127.044324732036,37.6202544613023,유료,https://culture.seoul.go.kr/culture/culture/cu...,"토요일 11:00 / 15:00 , 일요일 14:00"
